In [43]:
from google.colab import files
uploaded = files.upload()

Saving sales.csv to sales (1).csv


In [44]:
!pip install -q pyspark

In [45]:
import os
import shutil

In [46]:
# Create Data Lake directories
os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

In [47]:
# Copy original CSV to Bronze
shutil.copy(
    "/content/sales.csv",
    "/content/data_lake/bronze/sales.csv"
)

'/content/data_lake/bronze/sales.csv'

In [48]:
print("Data Lake structure created successfully!")

Data Lake structure created successfully!


In [49]:
#Creating a Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SalesDataLakeAssignment") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [50]:
#read the original bronze file
bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

print("Bronze data loaded successfully.")

Bronze data loaded successfully.


In [51]:
#Display the Schema
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [52]:
#diaplay top 10 rows
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [53]:
#total number of rows

before_count = df.count()

print("Total records before preprocessing:", before_count)

Total records before preprocessing: 1000


In [54]:
#checking for duplicate rows

before_count = df.count()

print("Total records before preprocessing:", before_count)

Total records before preprocessing: 1000


In [55]:
#Checking missing values

from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [56]:
#checking the text field

string_columns = [
    field.name
    for field in df.schema.fields
    if field.dataType.simpleString() == "string"
]

print("Text columns:")
print(string_columns)

Text columns:
['customer_id', 'customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_date', 'order_status']


In [57]:
#checking numeric column

numeric_columns = [
    field.name
    for field in df.schema.fields
    if field.dataType.simpleString() in ["int", "bigint", "double", "float"]
]

print("Numeric columns:")
print(numeric_columns)

Numeric columns:
['order_id', 'quantity', 'unit_price', 'discount_percent']


In [58]:
#checking the actual data carefully


df.show(20, truncate=False)

+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category   |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra  |Saree                 |Clothing   |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh     |T-Shirt               |Clothing   |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao     |Hoodie                |Clothing   |6       |3080      |12         

In [59]:
# Now we start creating the silver layer
# Now we removes the duplicates


silver_df = df.dropDuplicates()

In [60]:
#now we count the number of rows

print("Rows after removing exact duplicates:", silver_df.count())

Rows after removing exact duplicates: 990


In [61]:
# Now we trim spaces from the text

from pyspark.sql.functions import trim

for c in string_columns:
    silver_df = silver_df.withColumn(c, trim(col(c)))

In [62]:
# Converting the text in consistent format

from pyspark.sql.functions import initcap

for c in string_columns:
    silver_df = silver_df.withColumn(c, initcap(col(c)))

In [63]:
# Handling the missing text values

silver_df = silver_df.fillna(
    "Unknown",
    subset=string_columns
)

print("Missing text values handled.")

Missing text values handled.


In [64]:
#Handling the missing numeric values

from pyspark.sql.functions import avg

silver_df.select(
    avg("quantity").alias("avg_quantity"),
    avg("unit_price").alias("avg_unit_price"),
    avg("discount_percent").alias("avg_discount")
).show()

+-----------------+-----------------+------------+
|     avg_quantity|   avg_unit_price|avg_discount|
+-----------------+-----------------+------------+
|5.343434343434343|5723.919191919192|        19.9|
+-----------------+-----------------+------------+



In [65]:
# Applying the validation

from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "Quantity",
    when(col("Quantity") > 0, col("Quantity"))
    .otherwise(None)
)

In [66]:
# validating the unit prices

silver_df = silver_df.withColumn(
    "Unit_Price",
    when(col("Unit_Price") >= 0, col("Unit_Price"))
    .otherwise(None)
)

In [67]:
# validating the discount percentage

silver_df = silver_df.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)

In [68]:
# converting valid dates to consistent format

from pyspark.sql.functions import col, try_to_timestamp, lit, coalesce

silver_df = silver_df.withColumn(
    "order_date",
    coalesce(
        try_to_timestamp(col("order_date"), lit("dd/MM/yyyy")),
        try_to_timestamp(col("order_date"), lit("yyyy-MM-dd")),
        try_to_timestamp(col("order_date"), lit("MM/dd/yyyy"))
    ).cast("date")
)

In [69]:
# checking the cleaned Sliver DataFrame

silver_df.show(20, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |Quantity|Unit_Price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [70]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)



In [71]:
#count after preprocessing

after_count = silver_df.count()

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing:", after_count)

Rows before preprocessing: 1000
Rows after preprocessing: 990


In [73]:
# now saving the resulting data frame inside the Silver folder

silver_path = "/content/data_lake/silver/preprocessed_sales"

silver_df.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Silver data saved successfully.")

Silver data saved successfully.


In [74]:
# Save Silver as ONE CSV file
silver_export_path = "/content/silver_export"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_export_path)

print("Silver CSV generated!")

Silver CSV generated!


In [75]:
# Rename the Spark CSV to sales_silver.csv
import glob
import shutil

csv_file = glob.glob("/content/silver_export/part-*.csv")[0]

shutil.copy(csv_file, "/content/sales_silver.csv")

print("Created:")
print("/content/sales_silver.csv")

Created:
/content/sales_silver.csv


In [76]:
#downloading the csv file

from google.colab import files

files.download("/content/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>